# 12 -- Region accessors and Boolean DSL

venn-diagram-lab exposes four small helpers for selecting items and
regions inside an analysed diagram. The accessors return Python lists
of item identifiers; the Boolean DSL parses a short expression like
`A & B + B & C` into a list of region bitmasks; and `render_venn_svg()`
can draw the actual item names inside each region or spotlight a
subset of regions on top of the diagram. The four chain naturally --
this notebook walks through each surface on the bundled cancer
drivers sample, then composes them end-to-end.

**Audience:** researchers who already analyse with venn-diagram-lab
and want quick access to per-region item lists or presentation-ready
spotlight renderings.

**You will learn:**

1. The three accessors -- `intersection_items`, `exclusive_items`,
   `union_items` -- and three research questions they answer.
2. The Boolean DSL grammar (`&`, `+`/`|`, `~`/`!`, parentheses,
   atoms `A..I`) with one example per operator class.
3. `render_venn_svg(..., show_items=True)` to draw item identifiers
   inside each region.
4. `render_venn_svg(..., highlight=...)` to spotlight selected regions
   in label form or bitmask form.
5. A composability chain: DSL expression -> mask list -> spotlight render.
6. The same operations from the shell via `vdl data items`,
   `vdl data regions`, and `vdl render venn --highlight-expr`.
7. A decision table for picking the right pattern.


In [1]:
import venn_diagram_lab as vdl
from IPython.display import SVG, display

ds = vdl.load_sample('dataset_real_cancer_drivers_4')
res = vdl.analyze(ds)

print('Set inclusive sizes (items per source catalog):')
for name, size in res.set_sizes.items():
    print(f'  {name:14s} {size:5d}')


Set inclusive sizes (items per source catalog):
  Vogelstein       138
  COSMIC_CGC       581
  OncoKB          1231
  IntOGen          633


## 1. Three accessors -- one for each membership question

Three pure functions, one for each of the most common questions about
set membership:

| Function | Question it answers |
|---|---|
| `intersection_items(result, sets)` | Which items appear in EVERY set in `sets`? (regardless of other memberships) |
| `exclusive_items(result, sets)` | Which items appear in EXACTLY this combination? (and in no other set) |
| `union_items(result, sets)` | Which items appear in ANY of these sets? (deduplicated) |

All three accept either set letters (`'A'`, `'B'`, ...) or display
names (the values of `result.dataset.set_names`); the two forms may
be mixed. Output ordering follows `dataset.item_order` for
deterministic, reproducible results.


### Research question 1 -- the four-way consensus

*Which cancer-driver genes are shared by Vogelstein, COSMIC_CGC, and
OncoKB?* This uses `intersection_items` because the question allows
membership in IntOGen too -- we just want items present in all three
named catalogs.


In [2]:
shared_three = vdl.intersection_items(
    res, ['Vogelstein', 'COSMIC_CGC', 'OncoKB']
)
print(f'Shared by all three named catalogs: {len(shared_three)} items')
print('First 10:', shared_three[:10])


Shared by all three named catalogs: 126 items
First 10: ['ABL1', 'ACVR1B', 'AKT1', 'ALK', 'APC', 'AR', 'ARID1A', 'ARID1B', 'ARID2', 'ASXL1']


### Research question 2 -- catalog-exclusive drivers

*Which drivers does Vogelstein list that no other catalog reports?*
This uses `exclusive_items` -- items present in Vogelstein and in NONE
of COSMIC_CGC, OncoKB, IntOGen.


In [3]:
vogelstein_only = vdl.exclusive_items(res, ['Vogelstein'])
print(f'Vogelstein-only drivers: {len(vogelstein_only)}')
print(vogelstein_only)


Vogelstein-only drivers: 7
['FAM123B', 'H3F3A', 'HIST1H3B', 'MLL2', 'MLL3', 'MYCL1', 'SKP2']


### Research question 3 -- the candidate pool

*Across Vogelstein and OncoKB combined, how big is the candidate gene
list?* This uses `union_items` to deduplicate across the two
catalogs.


In [4]:
candidate_pool = vdl.union_items(res, ['Vogelstein', 'OncoKB'])
print(f'Union of Vogelstein and OncoKB: {len(candidate_pool)} unique drivers')
print('First 10 in item_order:', candidate_pool[:10])


Union of Vogelstein and OncoKB: 1238 unique drivers
First 10 in item_order: ['A1CF', 'AAMP', 'ABCB1', 'ABCC3', 'ABI1', 'ABL1', 'ABL2', 'ABRAXAS1', 'ACACA', 'ACKR3']
